In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import os
import seaborn as sns
import random
import csv
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [34]:
class CNNChromagram(nn.Module):
    def __init__(self, input_time = 861):
        super(CNNChromagram, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding = 'same', dilation = (1, 2)) # 1 x T x 12
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding = 'same', dilation  = (1, 2)) # 32 x T x 12
        self.conv3 = nn.Conv2d(64, 128, kernel_size = (3, 3), padding = 'same', dilation = (1, 2)) # 64 x T x 12

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 1))

        fc_input_size = 64 * (25 // 4)

        self.fc1 = nn.Linear(fc_input_size, 64)

        self.fc2 = nn.Linear(64, 4)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        batch_size, channels, freq_bins, time_steps = x.shape

        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, time_steps, -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [35]:
class ChromaDataset(Dataset):
    def __init__(self, audio_dir, label_dir, cache_dir, window_size=20, sr=16000, hop_length=512, num_classes = 4, target_bins=25):
        self.audio_dir = audio_dir
        self.label_dir = label_dir
        self.window_frames = window_size * sr // hop_length
        self.sr = sr
        self.num_classes = num_classes
        self.target_bins = target_bins
        self.audio_files = [f for f in os.listdir(audio_dir)]

        with open(label_dir, "r", encoding = "utf-8") as file:
            self.label_data = json.load(file)

        self.mode_mapping = {"Unkown": 0, "우조": 1, "계면조": 2, "아니리": 3, "창조": 0, "설렁제": 0}
        self.audio_segments = self.get_segments()
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok = True)
        self.cached_chromas = self.load_cached_chromas()

    def __len__(self):
        return len(self.audio_segments)

    def get_segments(self):
        segments = []
        for file_name in self.audio_files:
            file_path = os.path.join(self.audio_dir, file_name)
            duration = librosa.get_duration(filename = file_path)
            num_segments = int(duration // 20)
            start_times = np.random.uniform(0, max(1, duration - 20), num_segments)
            file_segments = [(file_name, start) for start in start_times]
            segments.extend(file_segments)

        return segments

    def extract_chroma(self, file_path):
        y, _ = librosa.load(file_path, sr=self.sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=self.sr)
        repeat = self.target_bins // chroma.shape[0] + 1
        expanded_chroma = np.tile(chroma, (repeat, 1))
        return expanded_chroma[:self.target_bins, :]

    def load_cached_chromas(self):
        cached_chromas = {}
        for file_name in self.audio_files:
            cache_path = os.path.join(self.cache_dir, file_name.replace("wav", ".npy"))
            if os.path.exists(cache_path):
                chroma = np.load(cache_path)
            else:
                chroma = self.extract_chroma(os.path.join(self.audio_dir, file_name))
                np.save(cache_path, chroma)

            cached_chromas[file_name] = chroma

        return cached_chromas

    def get_label(self, file_name, num_frames):
        hash_value = file_name.split("-")[0]
        labels = np.zeros((num_frames, self.num_classes))

        for item in self.label_data:
            if item.get("file_upload", "").startswith(hash_value):

                duration = librosa.get_duration(filename = os.path.join(self.audio_dir, file_name))

                for annotation in item.get("annotations", []):

                    for result in annotation.get("result", []):

                        value = result.get("value", {})
                        start_time, end_time = value.get("start", 0), value.get("end", 0)
                        mode_label = self.mode_mapping.get(value.get("labels", ["Unkwon"])[0], 0)

                        start_frame = int((start_time / duration) * num_frames)

                        end_frame = int((end_time / duration) * num_frames)

                        labels[start_frame : end_frame, mode_label] = 1

        return labels

    def __getitem__(self, idx):
        file_name, start_time = self.audio_segments[idx]
        file_path = os.path.join(self.audio_dir, file_name)
        chroma = self.cached_chromas[file_name]
        num_frames = chroma.shape[1]

        labels = self.get_label(file_name, num_frames)

        if num_frames < self.window_frames:
            chroma_pad = np.zeros((25, self.window_frames - num_frames))
            chroma = np.concatenate((chroma, chroma_pad), axis=1)
            label_pad = np.zeros((self.window_frames - num_frames, self.num_classes))
            labels = np.concatenate((labels, label_pad), axis=0)

        else:
            start_idx = np.random.randint(0, num_frames - self.window_frames)
            chroma = chroma[:, start_idx:start_idx + self.window_frames]
            labels = labels[start_idx:start_idx + self.window_frames]

        time = chroma.shape[1]

        labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(0)

        chroma_tensor = torch.tensor(chroma, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(labels, dtype=torch.float32)

        return chroma_tensor, label_tensor


In [ ]:
audio_dir = '/home/sangheon/Desktop/Pansori/Audio'
label_dir = '/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/label.json'
cache_dir = '/home/sangheon/Desktop/Pansori/cache'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = ChromaDataset(audio_dir, label_dir, cache_dir)
song_files = dataset.audio_files
loo = LeaveOneOut()
model = CNNChromagram().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

loo_losses = []
loo_accs = []
frame_accs = []

for train_idx, test_idx in loo.split(song_files):
    train_X, train_y = [], []
    for idx in train_idx:
        file_name = dataset.audio_files[idx]
        for segment in dataset.audio_segments:
            if segment[0] == file_name:
                segment_idx = dataset.audio_segments.index(segment)
                chroma, labels = dataset[segment_idx]
                train_X.append(chroma)
                train_y.append(labels)

    val_file = dataset.audio_files[test_idx[0]]
    val_X, val_y = [], []

    for segment in dataset.audio_segments:
        if segment[0] == val_file:
            segment_idx = dataset.audio_segments.index(segment)
            chroma, labels = dataset[segment_idx]
            val_X.append(chroma)
            val_y.append(labels)

    train_X = torch.stack(train_X).to(device)
    train_y = torch.stack(train_y).to(device)

    val_X = torch.stack(val_X).to(device)
    val_y = torch.stack(val_y).to(device)
    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(train_X)
        loss = criterion(outputs.permute(0, 2, 1), train_y.squeeze(1).argmax(dim=-1))
        loss.backward()
        optimizer.step()
    loo_losses.append(loss.item())

    model.eval()

    with torch.no_grad():
        val_output = model(val_X)
        val_loss = criterion(val_output.permute(0,2,1), val_y.squeeze(1).argmax(dim=-1))
        pred_probs = torch.softmax(val_output, dim = -1)
        pred_labels = torch.argmax(pred_probs, dim = -1)
        frame_accuracy = (pred_labels == val_y.squeeze(1).argmax(dim=-1)).float().mean(dim = 1).cpu().numpy()
        frame_accs.append(frame_accuracy)
        accuracy = frame_accuracy.mean().item()
        print(f"accuracy: {accuracy}")
    loo_losses.append(val_loss.item())
    loo_accs.append(accuracy)

print("\n LOO-CV Results")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accs):.4f}")

/tmp/ipykernel_738179/2561754917.py:27: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename = file_path)
/tmp/ipykernel_738179/2561754917.py:63: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename = os.path.join(self.audio_dir, file_name))
/tmp/ipykernel_738179/2561754917.py:105: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_tensor = torch.tensor(labels, dtype=torch.float32)


Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([251, 625, 4])
Outut shape: torch.Size([

KeyboardInterrupt: 